# 시나리오1 — CloudTrail 후보 선정과 근거 검증

교안 131~135페이지의 3주 집계·원시 로그 실습입니다. 여러 조사 후보의 우선순위를 정하고, 원시 근거와 분석 한계를 보고하는 것이 목표입니다. `02_장표사례_분석.ipynb`의 127페이지 요약 사건은 별개이며 동일 이름의 주체도 서로 연결하지 않습니다.

1. 학생 배포 ZIP을 PC에서 풉니다.
2. Colab의 **파일 → 노트북 업로드**로 이 파일을 엽니다. **Drive에 사본 저장**으로 개인 답안을 만듭니다.
3. **연결** 후 왼쪽 파일 창에 `cloudtrail_raw_events.csv`, `logs_aggregated_user_hour.csv`를 올립니다.
4. 아래 준비 셀부터 **Shift+Enter**로 실행합니다. 로컬 Jupyter는 노트북과 두 CSV를 같은 폴더에 둡니다.

데이터는 교육용 합성 CSV입니다. 기존 집계 재현용 `PassRole`, `ListBucket` 표기를 실제 CloudTrail API의 정확한 관찰로 인용하지 않습니다. 응답·키 식별자·정책 내용·MFA가 없는 부분은 미확인으로 보고하며, 그 필드 없이 끝낼 수 있는 과제만 필수로 구성했습니다. 실제 AWS 생성·경보 발송과 침해 피해량 확정은 이 파일의 완료 조건이 아닙니다.

기본 활동 M1~M6, 연결 실습 B1~B2, 심화 A1~A4, 최종 보고서 순서입니다. 설명 셀을 더블클릭해 예상을 적고 실행 후 판단을 수정합니다. 심화는 TODO 구현 후 정의 셀과 검증 셀을 다시 실행합니다. A1~A4 네 과제를 공통 목표로 진행합니다.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

from IPython.display import display
search_roots = [Path.cwd(), Path.cwd()/'학생용', Path.cwd().parent/'학생용']
if Path('/content').is_dir():
    search_roots.append(Path('/content'))
DATA = next((p for p in search_roots
             if (p / 'logs_aggregated_user_hour.csv').is_file()
             and (p / 'cloudtrail_raw_events.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Colab 파일 창에 CSV 두 개를 올리거나 로컬 노트북과 같은 폴더에 놓으세요.')

df = pd.read_csv(DATA / 'logs_aggregated_user_hour.csv')
raw = pd.read_csv(DATA / 'cloudtrail_raw_events.csv', keep_default_na=False)
raw['csv_line'] = np.arange(2, len(raw) + 2)
raw['time_kst'] = pd.to_datetime(raw['eventTime'], utc=True).dt.tz_convert('Asia/Seoul')
raw['hour_kst'] = raw['time_kst'].dt.strftime('%Y-%m-%d %H:00')
print(f'이벤트 {len(raw):,}행 / 집계 {len(df):,}행 / 분석 주체 {df.user.nunique()}개')
display(df.head(3))

## B1. 원시 이벤트에서 집계를 다시 계산하기

교안의 기준선 분석 전에 사용자·KST 시간창으로 두 자료를 연결합니다. 제공 집계의 숫자를 무조건 신뢰하거나 모든 특징을 0으로 나누지 않도록 점검하는 단계입니다.

아래 민감 API 사전은 **기존 합성 집계를 재현하기 위한 사전**입니다. 실제 운영 탐지 사전이 아닙니다. 오류율은 전체 호출을 분모로 계산해 소수 셋째 자리로 반올림합니다. 업무시간은 평일의 09~18시 시작 시간창(양 끝 포함)입니다.

실행 전 예상: 어떤 특징의 평시 중위수가 0 또는 결측일 수 있을까요?

In [ ]:
REPLAY_SENSITIVE = {'AttachRolePolicy','AttachUserPolicy','AuthorizeSecurityGroupIngress','CreateAccessKey','CreatePolicyVersion','CreateUser','PassRole','PutUserPolicy','SetDefaultPolicyVersion','UpdateAccessKey'}
replay = raw.assign(failed=raw.errorCode.ne(''), sensitive=raw.eventName.isin(REPLAY_SENSITIVE), login=raw.eventName.eq('ConsoleLogin'))
recomputed = replay.groupby(['user','hour_kst']).agg(
    api_call_count=('eventName','size'), error_rate=('failed','mean'), unique_regions=('awsRegion','nunique'),
    sensitive_api_count=('sensitive','sum'), unique_services=('eventSource','nunique'),
    source_ip_count=('sourceIPAddress','nunique'), console_login=('login','max')).reset_index().rename(columns={'hour_kst':'hour'})
recomputed['error_rate'] = recomputed.error_rate.round(3)
reconciliation = df.merge(recomputed,on=['user','hour'],how='outer',suffixes=('_provided','_recomputed'),indicator=True,validate='one_to_one')
check_columns = ['api_call_count','error_rate','unique_regions','sensitive_api_count','unique_services','source_ip_count','console_login']
quality_checks = pd.DataFrame({'field':check_columns,'different_rows':[int((reconciliation[c+'_provided'].astype(float)-reconciliation[c+'_recomputed'].astype(float)).abs().gt(1e-8).sum()) for c in check_columns]})
assert reconciliation['_merge'].eq('both').all(), '한쪽에만 있는 시간창이 있습니다. 데이터와 시간 변환을 확인하세요.'
assert quality_checks.different_rows.sum()==0, '원시와 집계의 차이를 확인한 뒤 진행하세요.'
display(quality_checks)
BASE_FEATURES = ['api_call_count','error_rate','unique_regions','sensitive_api_count','night_activity']
business = df[df.hour_of_day.between(9,18) & df.is_weekend.eq(0)]
baseline_table = business.groupby('user')[BASE_FEATURES].median().reindex(sorted(df.user.unique()))
display(baseline_table)
print('업무시간 기준선이 없는 주체:', baseline_table.index[baseline_table.api_call_count.isna()].tolist())
print('0을 분모로 쓰지 않습니다. 호출량은 중위수와 최소 80건, 다른 특징은 절대 조건을 함께 사용합니다.')

직접 확인: 사용자·시간창 한 개를 선택해 `raw[(raw.user == ...) & (raw.hour_kst == ...)]`를 출력합니다. 원시 행 수, 오류 수/전체 수, 리전 수를 제공 집계와 대조하고 원시 행 번호를 적으세요.

기준선 0과 기준선 결측은 다릅니다. 또한 업무시간의 평시와 매일 같은 야간 배치 시각의 평시는 다를 수 있습니다. 없는 시간창을 정상 0건으로 새로 채우지 않습니다.

## M1. 먼저 신고할 후보 고르기 · 8분

호출량이 높은 시간창 5개를 봅니다. 먼저 조사할 3개를 고르고 선택 이유를 적으세요.

실행 전 예상:  
선택한 후보와 이유:

In [ ]:
top_calls = df.sort_values(['api_call_count', 'user', 'hour'],
                           ascending=[False, True, True]).head(5)
display(top_calls[['user','hour','api_call_count','error_rate','unique_regions']])

이제 가장 호출이 많은 주체가 **같은 시각에 반복해서 작업하는지** 봅니다. 처음 결론을 수정할지 적으세요. 반복성이 있다고 해서 승인된 정상 작업이 확인된 것은 아닙니다.

판단 수정:  
작업 소유자에게 확인할 질문:

In [ ]:
subject = top_calls.iloc[0]['user']
clock_hour = int(top_calls.iloc[0]['hour_of_day'])
same_slot = df[(df.user == subject) & (df.hour_of_day == clock_hour)]
display(same_slot[['user','hour','api_call_count','error_rate',
                   'unique_regions','sensitive_api_count']].sort_values('hour'))
print('같은 시각의 관측 일수:', len(same_slot))
print('질문: 이 주체 전체를 조사에서 제외해도 될까요?')

## M2. 한 글자가 바꾸는 해시 · 7분

아래는 메모리 안에 만든 두 개의 교육용 기록입니다. CSV를 바꾸는 활동이 아닙니다.

실행 전 예상: 두 해시가 같을까요?  
실행 후 질문: 해시 일치는 사건의 진실성이나 수집 완전성까지 보장하나요? CloudTrail digest 검증과 이 활동은 어떻게 다를까요?

In [ ]:
sample_record = {'eventName':'CreateAccessKey', 'errorCode':'AccessDenied'}
changed_record = dict(sample_record, errorCode='')

def digest(record):
    text = json.dumps(record, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

display(pd.DataFrame([
    {'교육용 기록':'A', 'errorCode':sample_record['errorCode'], 'sha256':digest(sample_record)},
    {'교육용 기록':'B', 'errorCode':changed_record['errorCode'], 'sha256':digest(changed_record)},
]))
print('해시 일치:', digest(sample_record) == digest(changed_record))

## M3. 임계값을 바꾸면 경보가 몇 개 생길까 · 8분

아래는 **3개 신호로 하는 간단한 실험**이며, 학생 안내문의 5개 기준 규칙과는 별개입니다.

숫자를 한 개만 바꾸고 실행하기 전에 후보 수가 늘지 줄지 예상하세요. 같은 결과를 설명할 수 있는 정상 활동도 적습니다.

변경한 값:  
실행 전 예상:  
경보 수 감소만으로 품질이 좋아졌다고 말할 수 없는 이유:

In [ ]:
CALLS_MIN = 80
ERROR_MIN = 0.15
REGIONS_MIN = 3
MIN_SIGNALS = 2  # 1~3 중 선택

signals = pd.DataFrame({
    'calls': df.api_call_count >= CALLS_MIN,
    'errors': df.error_rate >= ERROR_MIN,
    'regions': df.unique_regions >= REGIONS_MIN,
})
trial = df.assign(signal_count=signals.sum(axis=1))
trial_selected = trial[trial.signal_count >= MIN_SIGNALS]
print('조사 후보 시간창 수:', len(trial_selected))
display(trial_selected[['user','hour','api_call_count','error_rate','unique_regions','signal_count']]
        .sort_values(['signal_count','api_call_count'], ascending=False).head(12))

## M4. 조사 예산이 5건이라면 · 10분

다음 셀은 두 모델과 기준 규칙을 한 번 계산합니다. 그 다음 셀에서는 **K만 바꿔** 바로 후보를 비교합니다.

원래 특징 모델은 contamination=0.01, log1p 모델은 0.005입니다. 이 둘의 경보 수만 보고 변환 효과를 결론 내리지 마세요. 변환 효과 검증은 contamination을 맞춘 별도 비교가 필요합니다.

모델은 `user`를 입력받지 않습니다. 특정 주체의 정책상 중요도도 스스로 알 수 없습니다. `decision_function`은 낮을수록 이상입니다.

In [ ]:
from sklearn.ensemble import IsolationForest

FEATURES = ['api_call_count','error_rate','unique_regions','sensitive_api_count','night_activity']
X = df[FEATURES].to_numpy(dtype=float)
X_log = X.copy()
X_log[:, 0] = np.log1p(X_log[:, 0])

model_raw = IsolationForest(n_estimators=300, contamination=0.01, random_state=42)
model_log = IsolationForest(n_estimators=300, contamination=0.005, random_state=42)
model_raw.fit(X)
model_log.fit(X_log)
scored_review = df.copy()
scored_review['if_score_raw'] = model_raw.decision_function(X)
scored_review['if_score_log'] = model_log.decision_function(X_log)
scored_review['if_flag_log'] = model_log.predict(X_log) == -1

bh = df[df.hour_of_day.between(9,18) & df.is_weekend.eq(0)]
median_calls = bh.groupby('user').api_call_count.median()
calls_base = df.user.map(median_calls)
scored_review['baseline_missing'] = calls_base.isna()
calls_base = calls_base.fillna(df.api_call_count.median())
rules = pd.DataFrame({
    'calls': df.api_call_count >= np.maximum(80, calls_base * 4),
    'errors': df.error_rate >= 0.15,
    'regions': df.unique_regions >= 3,
    'sensitive': df.sensitive_api_count >= 2,
    'night': df.night_activity.eq(1) &
             (df.sensitive_api_count.ge(1) | df.error_rate.ge(0.1) | df.api_call_count.ge(80)),
})
scored_review['rule_score'] = rules.sum(axis=1)
ranked = scored_review.sort_values(['if_score_log','user','hour']).reset_index(drop=True)
ranked['display_rank'] = np.arange(1,len(ranked)+1)
print('모델 계산 완료. 아래 셀에서는 K만 변경하세요.')

In [ ]:
K = 5  # 5, 20, 80을 비교하세요
if not 1 <= K <= len(ranked):
    raise ValueError('K는 1 이상 전체 행 수 이하로 입력하세요.')

top_k = ranked.head(K)
priority_intersection = ranked[(ranked.rule_score >= 3) & ranked.if_flag_log]
display(top_k[['display_rank','user','hour','rule_score','if_score_log','if_flag_log']].head(20))
print('선택한 시간창:', len(top_k))
print('선택한 시간창 중 기준 규칙 3점 이상:', int(top_k.rule_score.ge(3).sum()))
print('전체 자료에서 기준 규칙 3점 이상 AND 모델 경보:', len(priority_intersection))
print('K개 선택은 조사 예산입니다. 모델 경보 여부와 다를 수 있습니다.')

모델의 낮은 순위도 조사할 이유가 있을까요? 아래 주체를 확인하고 정책·업무 맥락에 따른 별도 판단을 적으세요.

추가 조사 이유:  
같은 특징값을 쓰는 두 방법의 교집합이 독립된 증거 두 개인지에 대한 판단:

In [ ]:
WATCH_SUBJECT = 'root'
display(ranked[ranked.user.eq(WATCH_SUBJECT)][
    ['display_rank','user','hour','api_call_count','sensitive_api_count',
     'console_login','rule_score','if_flag_log','baseline_missing']])

## B2. 조건을 맞춘 모델 비교와 상위 후보 표

M4의 두 모델은 기존 결과 재현을 위해 contamination이 다릅니다. 아래 비교는 **특징·학습 자료·시드·트리 수·contamination을 동일하게** 맞추고 변환만 바꿉니다. raw / 표준화 / 호출량 log1p를 같은 상위 K에서 비교합니다.

모든 행을 같은 자료에 학습·평가하므로 운영 탐지 성능의 검증은 아닙니다. 같은 경보 수가 같은 후보나 같은 품질을 의미하지 않습니다. 표준화만으로 IF 결과가 반드시 개선된다고 예상하지 마세요.

In [ ]:
from sklearn.preprocessing import StandardScaler
COMPARE_K = 20
assert 1 <= COMPARE_K <= len(df)
controlled_inputs = {'raw':X, 'standardized':StandardScaler().fit_transform(X), 'log1p_calls':X_log}
controlled_results = {}
controlled_sets = {}
compare_rows = []
for label, values in controlled_inputs.items():
    model = IsolationForest(n_estimators=300,contamination=0.01,random_state=42).fit(values)
    table = df[['user','hour']].copy()
    table['score'] = model.decision_function(values)
    table['flag'] = model.predict(values)==-1
    ordered = table.sort_values(['score','user','hour'])
    controlled_results[label] = ordered
    controlled_sets[label] = set(map(tuple,ordered.head(COMPARE_K)[['user','hour']].to_numpy()))
    compare_rows.append({'input':label,'contamination':0.01,'alerts':int(table.flag.sum()),'top_k':COMPARE_K})
controlled_summary = pd.DataFrame(compare_rows)
controlled_summary['overlap_with_raw'] = [len(controlled_sets[name] & controlled_sets['raw']) for name in controlled_summary.input]
display(controlled_summary)
rule_candidate_table = pd.concat([scored_review, rules.add_prefix('signal_')],axis=1).sort_values(['rule_score','api_call_count','user','hour'],ascending=[False,False,True,True]).head(5)
display(rule_candidate_table[['user','hour','rule_score'] + ['signal_'+c for c in rules.columns]])
print('상위 5개 표에서 후보·충족 신호·정상 대안·다음 확인 필드를 기록하세요.')

학생 과제: COMPARE_K를 5·20·80으로 바꾸어 후보 집합의 겹침을 비교하세요. 공통 후보와 차이 후보를 각각 하나 골라 원시 이벤트를 확인합니다. 이 코드는 공격 라벨이나 별도 점수 파일을 요구하지 않습니다.

기준 규칙 5점이 독립적인 증거 5개라는 뜻은 아닙니다. A4에서는 모델에 없는 중요 계정 정책을 별도로 보완합니다.

## M5. 키가 실제로 만들어졌을까 · 10분

먼저 03시의 키 생성·정책 부착 기록을 확인하세요. 성공/실패를 구별한 뒤 다음 셀의 04시 기록을 공개합니다.

03시의 관찰:  
04시를 본 뒤 바뀐 해석:  
생성된 키의 대상 사용자·키 식별자·권한 영향을 확인하려면 필요한 자료:

In [ ]:
FOCUS = 'dev07'
WINDOW = '2026-03-25 03:00'
actions = ['CreateAccessKey','AttachUserPolicy']
first = raw[(raw.user == FOCUS) & (raw.hour_kst == WINDOW)]
display(first[first.eventName.isin(actions)][
    ['csv_line','time_kst','user','eventName','sourceIPAddress','errorCode']])
display(first.groupby('sourceIPAddress').size().rename('events').reset_index())
print('이 시간창의 전체 호출:', len(first), '/ 오류 기록:', int(first.errorCode.ne('').sum()))

In [ ]:
next_window = raw[(raw.user == FOCUS) & (raw.hour_kst == '2026-03-25 04:00')]
display(next_window[next_window.eventName.isin(actions)][
    ['csv_line','time_kst','user','eventName','sourceIPAddress','errorCode']])
needed_fields = ['requestParameters','responseElements','accessKeyId','sessionContext']
print('이 CSV에 없는 추가 확인 필드:', [x for x in needed_fields if x not in raw.columns])

## M6. 그럴듯한 분석문 검증 · 8분

다음은 외부 모델을 호출한 결과가 아니라 **오류를 찾기 위해 작성한 교육용 가상 분석문**입니다.

> root가 21:49에 로그인한 다음 21:22에 자기 액세스 키를 만들었다. 로그인은 MFA 없이 성공했고, 따라서 외부 유출도 완료되었다.

각 문장을 `시간 모순`, `미제공 필드`, `과도한 결론`으로 검토하세요. 아래 표의 빈 오류 코드만으로 로그인 성공·MFA 여부를 판단하지 않습니다.

관찰 사실만 사용해 다시 쓴 문장:  
다음에 요청할 증거:

In [ ]:
root_window = raw[raw.user.eq('root') & raw.hour_kst.eq('2026-03-23 21:00')]
display(root_window[root_window.eventName.isin(['ConsoleLogin','CreateAccessKey'])]
        .sort_values('time_kst')[['csv_line','time_kst','userIdentityType','eventName','sourceIPAddress','errorCode']])
print('관찰 범위:', ', '.join(raw.columns[:9]))
print('판단 문장에는 기록에서 확인한 사실과 추가 확인 사항을 분리하세요.')

## 심화 실습 안내: 룰을 직접 완성하기

이제부터는 **문제 읽기 → 예상 기록 → TODO 구현 → 검증 셀 실행 → 원시 근거 확인** 순서입니다. A1~A4 네 과제가 공통 목표입니다. 진행이 느리면 강사의 단계별 힌트와 짝 검토를 활용합니다. 알고리즘을 새로 배우는 대신 Pandas 조건식·집계·결합으로 헌팅 룰을 만듭니다.

각 과제의 함수에서 `raise NotImplementedError(...)`를 지우고 자신의 코드를 작성하세요. 함수 정의 셀을 다시 실행한 뒤 바로 아래 검증 셀을 실행합니다. 수정 없이 실행하면 **미완료**만 표시됩니다. 검증 셀의 작은 가상 자료는 룰의 경계 조건을 확인하는 예시이며, 실제 사건의 정답 목록이 아닙니다. 통과해도 공격이 입증되는 것은 아닙니다.

검증 셀은 수정하지 않습니다. `AssertionError`가 나면 메시지의 조건을 확인하고 자신의 함수를 수정합니다. 통과를 위해 사용자·날짜·행 번호나 정답 표를 코드에 고정하지 마세요. 입력 DataFrame은 읽고, 결과는 새 객체로 반환합니다.

수업 순서에 따라 A1부터 먼저 이동해도 됩니다. **아래 공통 준비 셀을 한 번 실행**하세요. A1~A3은 첫 데이터 로딩 셀의 `df`, `raw`만 필요합니다. A4는 추가로 M4 모델 계산 셀이 필요합니다. 공통 준비 셀을 다시 실행하면 완료 표시가 초기화됩니다.

In [ ]:
advanced_status = {name: '미완료' for name in ['A1', 'A2', 'A3', 'A4']}
print('심화 준비 완료. 함수 구현 전에는 각 과제가 미완료로 표시됩니다.')

## A1. 정기 배치의 예외를 좁게 만들기 · 20분

**가설:** 호출량이 많다는 이유로 후보가 된 정기 배치를 좁게 제외하면 조사 부담을 줄일 수 있다. 단, 같은 계정의 다른 활동을 함께 숨기면 안 된다.

실습용 업무 확인 카드: 담당자가 `ci-deploy`의 **매일 03시(KST)** 배치를 확인했다고 가정합니다. 이 카드는 연습을 위한 가정이며 CSV가 업무 승인을 증명하지는 않습니다. 다음 조건을 **모두** 만족할 때만 호출량 후보에서 예외 처리합니다.

| 항목 | 예외 조건 |
|---|---|
| 계정 | `user == 'ci-deploy'` |
| 시각 | `hour_of_day == 3` |
| 리전 수 | `unique_regions == 1` |
| 오류율 | `error_rate < 0.02` (2% 미만) |
| 민감 API | `sensitive_api_count == 0` |

구현할 `batch_exception_mask(table)`은 입력과 **같은 인덱스의 bool Series**를 반환합니다. `True`는 이 호출량 룰의 예외입니다. 다른 룰이나 원시 조사에서 이 계정을 삭제한다는 뜻은 아닙니다. 호출량 후보 선정과 예외 적용은 검증 셀이 제공합니다.

실행 전 예상 — `user != 'ci-deploy'`만 사용하면 놓칠 수 있는 상황:

<details><summary>힌트 1: 조건식</summary>
각 열을 비교하면 행별 True/False가 나옵니다. 조건을 괄호로 감싸고 `&`로 연결하세요. Python의 `and` 대신 Pandas의 `&`를 씁니다.
</details>
<details><summary>힌트 2: 경계값</summary>
2%와 2% 미만은 다릅니다. 반환 직전에 결과의 길이·인덱스·dtype을 확인하세요.
</details>

In [ ]:
advanced_status['A1'] = '미완료'
def batch_exception_mask(table):
    # TODO: 다섯 조건을 모두 만족하는 행만 True인 bool Series를 반환하세요.
    raise NotImplementedError('A1: 정기 배치의 예외 조건을 구현하세요.')

In [ ]:
advanced_status['A1'] = '미완료'
a1_test = pd.DataFrame([
    ['ci-deploy', 3, 1, 0.019, 0],  # 카드의 모든 조건 충족
    ['ci-deploy', 5, 1, 0.010, 0],  # 다른 시각
    ['ci-deploy', 3, 2, 0.010, 0],  # 다른 리전 범위
    ['ci-deploy', 3, 1, 0.020, 0],  # 오류율 경계
    ['ci-deploy', 3, 1, 0.010, 1],  # 민감 API 존재
    ['other-batch', 3, 1, 0.010, 0],  # 다른 계정
], columns=['user', 'hour_of_day', 'unique_regions', 'error_rate', 'sensitive_api_count'],
   index=[11, 13, 17, 19, 23, 29])
try:
    a1_answer = batch_exception_mask(a1_test.copy())
    assert isinstance(a1_answer, pd.Series), 'bool Series를 반환하세요.'
    assert a1_answer.index.equals(a1_test.index), '원래 인덱스를 유지하세요.'
    assert pd.api.types.is_bool_dtype(a1_answer) and not a1_answer.isna().any(), '빈 값 없는 bool Series가 필요합니다.'
    assert a1_answer.tolist() == [True, False, False, False, False, False], '계정 전체를 제외하지 말고 다섯 조건과 2% 경계를 확인하세요.'
    a1_candidates = df[df.api_call_count.ge(80)].copy()
    a1_mask = batch_exception_mask(a1_candidates.copy())
    assert a1_mask.index.equals(a1_candidates.index), '실제 자료에서도 인덱스를 유지하세요.'
    a1_remaining = a1_candidates.loc[~a1_mask]
    print('호출량 후보:', len(a1_candidates), '/ 예외:', int(a1_mask.sum()), '/ 남은 후보:', len(a1_remaining))
    print('예외 적용 뒤에도 남아 있는 같은 계정의 활동:')
    display(a1_remaining[a1_remaining.user.eq('ci-deploy')][
        ['user', 'hour', 'api_call_count', 'error_rate', 'unique_regions', 'sensitive_api_count']])
    advanced_status['A1'] = '통과'
    print('A1 코드 검증 통과. 아래 해석까지 작성하세요.')
except NotImplementedError as pending:
    print('미완료 —', pending)

A1 해석:

- 후보 수: 적용 전 __ / 예외 __ / 적용 후 __
- 같은 계정에서 남은 시간창과 예외로 처리되지 않은 이유:
- 업무 확인 카드 없이 자동으로 예외 처리하면 생기는 위험:
- 추가 실험: **메모리의 복사본**에서 배치 한 행의 민감 API 수를 1로 바꾸고 예외가 해제되는지 확인하는 코드를 새 셀에 작성하세요. CSV를 저장하거나 수정하지 않습니다.

## A2. IAM 조회 거부가 집중된 시간창 찾기 · 25분

**가설:** 여러 IAM 조회 API에서 권한 거부가 집중되면 권한 범위를 탐색한 흔적일 수 있다. 정상적인 권한 점검이나 설정 오류라는 대안 설명도 검토한다.

`hunt_iam_denials(events)`를 완성하세요. 입력은 `raw`와 같은 열을 가진 원시 이벤트 표입니다.

1. `eventSource == 'iam.amazonaws.com'`이고 `eventName`이 `List` 또는 `Get`으로 시작하는 **IAM 조회 호출**만 고릅니다.
2. `user`, `hour_kst`별로 집계합니다. 오류율의 분모는 **그 사용자·그 시간의 IAM 조회 호출 전체**입니다. 다른 서비스·다른 시간의 호출을 섞지 않습니다.
3. 다음 세 조건을 모두 만족하는 시간창만 반환합니다: **AccessDenied 5건 이상**, **거부율 50% 이상**, **거부된 서로 다른 API 3종 이상**.

반환 열: `user`, `hour_kst`, `enum_calls`(조회 전체), `denied_count`(AccessDenied), `denied_rate`(0~1), `denied_api_types`(거부된 API 종류 수). 후보가 없어도 이 열을 가진 빈 DataFrame을 반환합니다.

실행 전 예상 — 오류 수만 세면 발생하는 오탐:

<details><summary>힌트 1: 필터와 집계</summary>
문자열 접두어는 `.str.startswith(('List', 'Get'))`, 여러 열별 집계는 `.groupby([...])`로 처리할 수 있습니다. bool 열의 합은 True의 개수입니다.
</details>
<details><summary>힌트 2: 서로 다른 API 수</summary>
`.nunique()`를 적용할 대상은 전체 호출이 아니라 **AccessDenied 호출의 eventName**입니다. 두 집계 표는 사용자와 시간 키로 결합하고 없는 거부 집계는 0으로 채우세요.
</details>

In [ ]:
advanced_status['A2'] = '미완료'
def hunt_iam_denials(events):
    # TODO 1: IAM의 List/Get 호출을 선택하세요.
    # TODO 2: 사용자·시간별 분모, AccessDenied 수, 거부된 API 종류 수를 구하세요.
    # TODO 3: 거부율을 계산하고 세 임계 조건을 모두 적용해 결과 표를 반환하세요.
    raise NotImplementedError('A2: IAM 조회 거부를 집계하는 룰을 구현하세요.')

In [ ]:
advanced_status['A2'] = '미완료'
# 작은 가상 자료: 서로 다른 시간·서비스가 분모에 섞이는지 확인합니다.
a2_rows = []
def add_iam_test(subject, hour, denied, total, kinds, source='iam.amazonaws.com'):
    names = ['ListUsers', 'GetRole', 'ListRoles'][:kinds]
    for i in range(total):
        a2_rows.append({'user': subject, 'hour_kst': hour, 'eventSource': source,
                        'eventName': names[i % len(names)],
                        'errorCode': 'AccessDenied' if i < denied else ''})
add_iam_test('edge', '10:00', 5, 10, 3)  # 세 조건의 포함 경계
add_iam_test('edge', '11:00', 0, 20, 3)  # 같은 사용자의 다른 시간
add_iam_test('low-rate', '10:00', 5, 11, 3)
add_iam_test('low-count', '10:00', 4, 4, 3)
add_iam_test('few-apis', '10:00', 6, 6, 2)
add_iam_test('wrong-source', '10:00', 6, 6, 3, 'ec2.amazonaws.com')
add_iam_test('edge', '10:00', 0, 20, 3, 'ec2.amazonaws.com')
for i in range(6):
    a2_rows.append({'user': 'wrong-action', 'hour_kst': '10:00',
                    'eventSource': 'iam.amazonaws.com', 'eventName': ['CreateUser', 'DeleteUser', 'UpdateUser'][i % 3],
                    'errorCode': 'AccessDenied'})
# 성공한 별도 API를 거부 API의 종류 수에 더하지 않는지 확인합니다.
add_iam_test('successful-diversity', '10:00', 5, 5, 1)
for action in ['GetRole', 'ListRoles']:
    a2_rows.append({'user': 'successful-diversity', 'hour_kst': '10:00',
                    'eventSource': 'iam.amazonaws.com', 'eventName': action, 'errorCode': ''})
for i in range(6):
    a2_rows.append({'user': 'other-error', 'hour_kst': '10:00', 'eventSource': 'iam.amazonaws.com',
                    'eventName': ['ListUsers', 'GetRole', 'ListRoles'][i % 3], 'errorCode': 'ThrottlingException'})
a2_test = pd.DataFrame(a2_rows)
try:
    a2_answer = hunt_iam_denials(a2_test.copy())
    a2_columns = ['user', 'hour_kst', 'enum_calls', 'denied_count', 'denied_rate', 'denied_api_types']
    assert isinstance(a2_answer, pd.DataFrame) and set(a2_columns).issubset(a2_answer.columns), '지정된 여섯 열을 반환하세요.'
    assert len(a2_answer) == 1 and set(zip(a2_answer.user, a2_answer.hour_kst)) == {('edge', '10:00')}, '집계 키, IAM 조회 필터, 거부 API 종류 수, 세 임계값을 확인하세요.'
    a2_row = a2_answer.iloc[0]
    assert a2_row.enum_calls == 10 and a2_row.denied_count == 5 and a2_row.denied_api_types == 3, '분모와 AccessDenied 집계가 다릅니다.'
    assert abs(a2_row.denied_rate - 0.5) < 1e-9, '거부율은 해당 시간의 IAM 조회 호출을 분모로 계산합니다.'
    a2_empty = hunt_iam_denials(a2_test.iloc[:0].copy())
    assert a2_empty.empty and set(a2_columns).issubset(a2_empty.columns), '후보가 없을 때도 반환 열을 유지하세요.'
    a2_hits = hunt_iam_denials(raw.copy())
    print('IAM 조회 거부 후보 시간창:', len(a2_hits))
    display(a2_hits.sort_values(['denied_rate', 'denied_count'], ascending=False))
    advanced_status['A2'] = '통과'
    print('A2 코드 검증 통과. 원시 근거와 정상 대안 설명을 추가하세요.')
except NotImplementedError as pending:
    print('미완료 —', pending)

A2 해석:

- 후보 한 건과 IAM 조회 분모·거부 수·거부율:
- 오류가 많다는 사실만으로 악의적인 조회라고 확정할 수 없는 이유:
- **직접 코드 추가:** 후보 하나의 사용자·시간을 골라 해당 IAM 조회의 `csv_line`, `eventName`, `sourceIPAddress`, `errorCode`를 출력하세요.
- 룰 변경안 한 가지와 예상 효과(정상 점검 업무가 더 걸릴까, 탐색 흔적을 덜 볼까?):

## A3. 실패 뒤의 후속 호출을 시간으로 연결하기 · 30분

**가설:** 키 생성 거부 이후 가까운 시각의 오류 없는 키 생성 호출을 함께 보면 후속 확인 대상을 좁힐 수 있다. 같은 IP는 같은 행위자라는 증명이 아니다.

`correlate_key_calls(events, window_minutes=60)`의 준비 코드는 제공합니다. **결합 기준과 시간 조건**을 직접 구현하세요.

- 대상은 `CreateAccessKey`입니다. 앞 이벤트는 `errorCode == 'AccessDenied'`, 뒤 이벤트는 `errorCode == ''`입니다.
- 사용자 **AND 출발지 IP가 모두 같아야** 연결합니다. 다른 계정이나 다른 IP는 별도 후보입니다.
- `0 < 뒤 시각 - 앞 시각 <= 60분`을 적용합니다. 같은 시각·역순·60분 초과는 제외합니다.
- 한 실패에 후속 호출이 여럿이면 **가장 이른 후속 호출 1개**를 남깁니다. 동일 시각이면 `followup_line`이 작은 행을 선택합니다. 결과의 단위는 실패 이벤트별 연결 후보이며, 하나의 후속 이벤트가 여러 실패와 연결될 수 있습니다.
- 반환 열: `user`, `sourceIPAddress`, `failed_line`, `followup_line`, `failed_time`, `followup_time`, `gap_minutes`. CSV 행 번호는 근거를 찾는 위치입니다.

실행 전 예상 — 같은 사용자만 연결하거나 시간 차이의 절댓값을 사용하면 생기는 오류:

<details><summary>힌트 1: 결합</summary>
`before.merge(after, on=[...])`의 키를 생각하세요. 준비 코드에서 실패와 후속 호출의 시각·행 번호는 이미 다른 이름으로 바꾸었습니다.
</details>
<details><summary>힌트 2: 시간 조건과 중복</summary>
두 시각 Series를 빼고 `.dt.total_seconds() / 60`으로 분 단위 차이를 계산할 수 있습니다. 범위를 제한한 뒤 `.sort_values(...)`, `.drop_duplicates(subset=[...], keep='first')`로 실패별 첫 후속 호출을 남겨 보세요.
</details>

In [ ]:
advanced_status['A3'] = '미완료'
def correlate_key_calls(events, window_minutes=60):
    columns = ['user', 'sourceIPAddress', 'time_kst', 'errorCode', 'csv_line']
    calls = events.loc[events.eventName.eq('CreateAccessKey'), columns].copy()
    before = calls[calls.errorCode.eq('AccessDenied')].drop(columns='errorCode').rename(
        columns={'csv_line': 'failed_line', 'time_kst': 'failed_time'})
    after = calls[calls.errorCode.eq('')].drop(columns='errorCode').rename(
        columns={'csv_line': 'followup_line', 'time_kst': 'followup_time'})
    # TODO 1: 사용자와 IP가 모두 같은 이벤트끼리 결합하세요.
    # TODO 2: gap_minutes를 계산하고 0 < gap <= window_minutes 조건을 적용하세요.
    # TODO 3: 실패별 가장 이른 후속 호출을 선택하고 지정된 일곱 열을 반환하세요.
    raise NotImplementedError('A3: 결합 키와 순방향 시간 조건을 구현하세요.')

In [ ]:
advanced_status['A3'] = '미완료'
a3_test = pd.DataFrame([
    [1, 'a', '192.0.2.1', '10:00', 'AccessDenied', 'CreateAccessKey'],
    [2, 'a', '192.0.2.1', '09:59', '', 'CreateAccessKey'],
    [3, 'a', '192.0.2.2', '10:01', '', 'CreateAccessKey'],
    [4, 'b', '192.0.2.1', '10:02', '', 'CreateAccessKey'],
    [5, 'a', '192.0.2.1', '10:00', '', 'CreateAccessKey'],
    [6, 'a', '192.0.2.1', '10:20', 'AccessDenied', 'ListUsers'],
    [7, 'a', '192.0.2.1', '10:15', '', 'ListUsers'],
    [8, 'a', '192.0.2.1', '10:25', 'ThrottlingException', 'CreateAccessKey'],
    [9, 'a', '192.0.2.1', '10:30', '', 'CreateAccessKey'],
    [10, 'a', '192.0.2.1', '10:30', '', 'CreateAccessKey'],
    [11, 'a', '192.0.2.1', '10:40', '', 'CreateAccessKey'],
    [12, 'c', '192.0.2.3', '12:00', 'AccessDenied', 'CreateAccessKey'],
    [13, 'c', '192.0.2.3', '13:00', '', 'CreateAccessKey'],
    [14, 'd', '192.0.2.4', '12:00', 'AccessDenied', 'CreateAccessKey'],
    [15, 'd', '192.0.2.4', '13:01', '', 'CreateAccessKey'],
], columns=['csv_line', 'user', 'sourceIPAddress', 'clock', 'errorCode', 'eventName'])
a3_test['time_kst'] = pd.to_datetime('2026-01-01 ' + a3_test.clock).dt.tz_localize('Asia/Seoul')
# 입력 순서에 의존하지 않고 시각과 행 번호로 선택하는지 확인합니다.
a3_test = a3_test.sample(frac=1, random_state=42)
try:
    a3_answer = correlate_key_calls(a3_test.copy(), window_minutes=60)
    a3_columns = ['user', 'sourceIPAddress', 'failed_line', 'followup_line', 'failed_time', 'followup_time', 'gap_minutes']
    assert isinstance(a3_answer, pd.DataFrame) and set(a3_columns).issubset(a3_answer.columns), '지정된 일곱 열을 반환하세요.'
    assert len(a3_answer) == 2 and set(zip(a3_answer.failed_line, a3_answer.followup_line)) == {(1, 9), (12, 13)}, '같은 사용자·IP, 순방향 시간, 60분 경계, 첫 후속 호출을 확인하세요.'
    assert a3_answer.set_index('failed_line').gap_minutes.to_dict() == {1: 30.0, 12: 60.0}, '시각 차이를 분 단위로 계산하세요.'
    a3_short = correlate_key_calls(a3_test.copy(), window_minutes=20)
    assert a3_short.empty, 'window_minutes 인수를 실제 시간 조건에 사용하세요.'
    a3_empty = correlate_key_calls(a3_test.iloc[:0].copy())
    assert a3_empty.empty and set(a3_columns).issubset(a3_empty.columns), '연결 대상이 없어도 반환 열을 유지하세요.'
    a3_pairs = correlate_key_calls(raw.copy(), window_minutes=60)
    print('실패 이후 오류 없는 후속 호출의 연결 후보:', len(a3_pairs))
    display(a3_pairs.sort_values(['failed_time', 'failed_line']))
    advanced_status['A3'] = '통과'
    print('A3 코드 검증 통과. 대상 키와 권한 영향은 별도 확인이 필요합니다.')
except NotImplementedError as pending:
    print('미완료 —', pending)

A3 해석:

- 연결된 두 원시 CSV 행 번호와 시간 차이:
- **직접 코드 추가:** `window_minutes`를 15, 30, 60으로 바꿔 연결 후보 수를 표로 비교하세요. 0건이라는 결과도 기록합니다.
- 같은 사용자·IP를 연결했는데도 같은 키·같은 세션이라고 확정할 수 없는 이유:
- `오류 없는 후속 호출`을 확인한 사실과, 생성 대상·MFA·실제 권한 영향을 확인하기 위해 필요한 추가 필드를 구분해 작성하세요.

## A4. 모델 경보에서 빠진 중요 계정 활동 보완하기 · 15분

선행 셀: **M4 모델 계산**. `scored_review`가 없으면 M4부터 실행하세요.

**가설:** 일반적인 빈도 이상 탐지와 별개로, 중요 계정의 특정 활동은 검토 후보에 포함할 필요가 있다. 이번 가상 조직은 root의 로그인·키 생성 호출을 모두 사람이 확인한다는 정책을 갖는다고 가정한다.

준비 셀은 원시 자료의 `userIdentityType == 'Root'`이면서 `ConsoleLogin` 또는 `CreateAccessKey`인 이벤트를 사용자·KST 시간창별로 세어 `root_action_count`로 붙입니다. 원시 근거도 `a4_evidence`에 남깁니다.

`combine_priorities(table)`에서 입력과 같은 인덱스를 가진 DataFrame에 아래 bool 열 세 개를 만들어 반환하세요.

| 반환 열 | 정책 |
|---|---|
| `base_priority` | `if_flag_log`가 True **AND** `rule_score >= 3` |
| `critical_identity` | `root_action_count >= 1` |
| `selected` | 위 두 조건 중 **어느 하나라도** True |

원래 점수나 모델 결과를 바꾸지 않고 최종 선정 정책을 구현합니다. root라는 이름만으로 모든 시간창을 선택하거나 모델 경보만 무조건 선택하지 않습니다.

실행 전 예상 — 두 정책을 AND로만 연결하면 생기는 누락:

<details><summary>힌트</summary>
`.ge(...)`, `&`, `|`를 사용하세요. 마지막 열은 먼저 만든 두 bool 열을 조합합니다. 선정 이유를 두 열로 남기면 왜 후보가 되었는지 설명할 수 있습니다.
</details>

In [ ]:
advanced_status['A4'] = '미완료'
def combine_priorities(table):
    result = pd.DataFrame(index=table.index)
    # TODO: base_priority, critical_identity, selected의 세 bool 열을 작성하세요.
    raise NotImplementedError('A4: 기본 경보와 중요 계정 정책을 결합하세요.')

In [ ]:
advanced_status['A4'] = '미완료'
a4_test = pd.DataFrame({
    'if_flag_log': [True, True, False, False, False, True],
    'rule_score': [3, 2, 4, 0, 0, 4],
    'root_action_count': [0, 0, 0, 1, 0, 2],
}, index=[11, 13, 17, 19, 23, 29])
try:
    a4_answer = combine_priorities(a4_test.copy())
    assert isinstance(a4_answer, pd.DataFrame) and a4_answer.index.equals(a4_test.index), '입력과 같은 인덱스의 DataFrame을 반환하세요.'
    for col in ['base_priority', 'critical_identity', 'selected']:
        assert col in a4_answer and pd.api.types.is_bool_dtype(a4_answer[col]) and not a4_answer[col].isna().any(), f'{col}은 빈 값 없는 bool 열이어야 합니다.'
    assert a4_answer.base_priority.tolist() == [True, False, False, False, False, True], '기본 경보의 AND 조건과 3점 경계를 확인하세요.'
    assert a4_answer.critical_identity.tolist() == [False, False, False, True, False, True], '중요 계정의 해당 활동 유무를 확인하세요.'
    assert a4_answer.selected.tolist() == [True, False, False, True, False, True], '두 정책 중 하나만 충족해도 최종 후보입니다.'
    a4_evidence = raw[raw.userIdentityType.eq('Root') & raw.eventName.isin(['ConsoleLogin', 'CreateAccessKey'])].copy()
    a4_counts = a4_evidence.groupby(['user', 'hour_kst']).size().rename('root_action_count').reset_index().rename(columns={'hour_kst': 'hour'})
    a4_input = scored_review.merge(a4_counts, on=['user', 'hour'], how='left', validate='one_to_one')
    a4_input['root_action_count'] = a4_input.root_action_count.fillna(0).astype(int)
    a4_flags = combine_priorities(a4_input.copy())
    assert a4_flags.index.equals(a4_input.index), '실제 자료에서도 인덱스를 유지하세요.'
    a4_result = pd.concat([a4_input, a4_flags], axis=1)
    print('기본 정책 후보:', int(a4_flags.base_priority.sum()), '/ 결합 후:', int(a4_flags.selected.sum()))
    display(a4_result[a4_result.selected][['user', 'hour', 'rule_score', 'if_flag_log', 'root_action_count', 'base_priority', 'critical_identity']])
    print('중요 계정 정책의 원시 근거:')
    display(a4_evidence[['csv_line', 'time_kst', 'userIdentityType', 'eventName', 'errorCode']])
    advanced_status['A4'] = '통과'
    print('A4 코드 검증 통과. 선정 이유와 정책의 한계를 작성하세요.')
except NotImplementedError as pending:
    print('미완료 —', pending)

A4 해석:

- 기본 정책 후보 수와 결합 후 후보 수:
- 모델 경보가 없지만 추가로 선정된 시간창·원시 근거:
- 중요 계정 후보로 선정된 사실과 공격으로 판정하는 일의 차이:
- **확장:** 이 보완책으로도 놓칠 수 있는 행위 가설을 하나 제안하고 필요한 로그 필드를 적으세요.

## 심화 완료 확인

아래 상태는 검증 셀 통과 여부입니다. 해석·직접 추가한 코드의 적절성은 짝 또는 강사가 검토합니다. A1~A4의 **구현 코드 + 검증 출력 + 해석**을 제출하세요. A4는 보완 정책의 한계까지 설명합니다.

함수를 수정하면 정의 셀과 검증 셀을 다시 실행한 뒤 완료 확인 셀을 실행합니다. 저장된 출력만 남아 있는 상태는 현재 런타임의 검증 완료를 의미하지 않습니다.

In [ ]:
display(pd.DataFrame({'과제': list(advanced_status), '코드 검증': list(advanced_status.values())}))
required_done = all(advanced_status[name] == '통과' for name in ['A1', 'A2', 'A3', 'A4'])
print('필수 코드 검증:', '완료 — 해석과 추가 코드도 확인하세요.' if required_done else '미완료 — TODO 구현 후 각 검증 셀을 실행하세요.')

## 최종 제출 — 데이터만으로 완성하는 헌팅 결과서

가설 2개, 기준선·시간창 메모, 조사 후보 5개, 비교표, A1~A4 구현·검증·해석을 제출합니다. 선택 사건 하나를 아래 형식으로 보고합니다. 별도 강사 라벨·AWS 계정·외부 모델은 필요하지 않습니다.

| 항목 | 답안 |
|---|---|
| 조사 질문·사용자·시간창 | |
| 규칙 및 모델 후보 선정 이유 | |
| 원시 근거 3개 이상(csv_line·시각·이벤트·오류) | |
| 확인한 사실과 정상 대안 | |
| 확인할 수 없는 대상·권한·MFA·영향과 필요한 필드 | |
| 후속 조치 및 추가 수집 제안 | |
| 동료가 지적한 비약과 수정 문장 | |

`미확인`을 근거와 함께 적는 것도 완료된 분석입니다. 없는 응답·키·정책을 얻을 때까지 진행을 멈추지 않습니다. 생성 라벨을 이용한 정밀도/재현율은 강사의 종료 평가 확장이며 필수 제출 조건이 아닙니다.